## **DO NOT rename or change the signature of these functions. Your code must be in the 3rd cell of the notebook, otherwise the tests will fall.**

# Homework: AI Agents

## Instructions
1. **"Template" cell** — run it first, do not modify.
2. **"Tasks" cell** — write your code where you see `# YOUR CODE HERE`.
3. Run the open examples and make sure all say `OK`.
4. Submit the notebook with saved outputs.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          TEMPLATE — DO NOT MODIFY THIS CELL                 ║
# ╚══════════════════════════════════════════════════════════════╝
%pip install -q langchain-openai langchain-core

import os, json, copy
from typing import Any
from pathlib import Path
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

MODEL_NAME = "gpt-oss-20b"
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "YOUR_KEY_HERE")
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)


def llm_chat(messages: list, tools: list | None = None) -> AIMessage:
    """
    Sends the message history to the LLM and returns the model response.

    Parameters:
      messages — list of dialog messages. Each message is a LangChain object:
                   SystemMessage(content="...")   — instruction for the model (agent role)
                   HumanMessage(content="...")    — message from the user
                   AIMessage(...)                 — previous model response
                   ToolMessage(content="...", tool_call_id="...") — tool result

      tools   — list of tool descriptions (OpenAI function calling schema or LangChain tools).

    Returns AIMessage:
      msg.content    — text response (str)
      msg.tool_calls — list of tool calls:
                         "name" — tool name
                         "args" — arguments (already parsed dict)
                         "id"   — unique call identifier
    """
    if tools:
        return llm.bind_tools(tools).invoke(messages)
    return llm.invoke(messages)


# Product catalog
CATALOG = [
    {"id": "p1",  "name": "Sony WH-1000XM5",            "category": "headphones", "brand": "Sony",     "price": 349, "color": "black",    "rating": 4.8, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p2",  "name": "Sony WH-CH720N",              "category": "headphones", "brand": "Sony",     "price": 129, "color": "blue",     "rating": 4.4, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p3",  "name": "Bose QuietComfort Ultra",     "category": "headphones", "brand": "Bose",     "price": 379, "color": "white",    "rating": 4.7, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p4",  "name": "Apple AirPods Pro 2",         "category": "earbuds",    "brand": "Apple",    "price": 249, "color": "white",    "rating": 4.6, "tags": ["wireless", "noise-cancelling", "ios"]},
    {"id": "p5",  "name": "Anker Soundcore Liberty 4 NC","category": "earbuds",    "brand": "Anker",    "price": 99,  "color": "black",    "rating": 4.3, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p6",  "name": "Logitech MX Master 3S",       "category": "mouse",      "brand": "Logitech", "price": 109, "color": "graphite", "rating": 4.8, "tags": ["wireless", "productivity", "premium"]},
    {"id": "p7",  "name": "Logitech Pebble 2",           "category": "mouse",      "brand": "Logitech", "price": 34,  "color": "white",    "rating": 4.2, "tags": ["wireless", "budget", "portable"]},
    {"id": "p8",  "name": "Keychron K2",                 "category": "keyboard",   "brand": "Keychron", "price": 89,  "color": "black",    "rating": 4.5, "tags": ["wireless", "mechanical", "compact"]},
    {"id": "p9",  "name": "NuPhy Air75",                 "category": "keyboard",   "brand": "NuPhy",    "price": 139, "color": "gray",     "rating": 4.6, "tags": ["wireless", "mechanical", "low-profile"]},
    {"id": "p10", "name": "Amazon Kindle Paperwhite",    "category": "ereader",    "brand": "Amazon",   "price": 149, "color": "black",    "rating": 4.7, "tags": ["reading", "portable", "gift"]},
]


@dataclass
class ShopState:
    """Session state: cart and last search results."""
    cart: list = field(default_factory=list)
    last_results: list = field(default_factory=list)


@dataclass
class ToolCallRecord:
    name: str
    args: dict
    result: Any = None


class ToolTracer:
    """Collects all tool calls."""
    def __init__(self):
        self.calls: list[ToolCallRecord] = []

    def record(self, name: str, args: dict, result: Any = None) -> None:
        self.calls.append(ToolCallRecord(name=name, args=args, result=result))

    def called(self, name: str) -> bool:
        return any(c.name == name for c in self.calls)

    def get_calls(self, name: str) -> list:
        return [c for c in self.calls if c.name == name]

    def print_trace(self) -> None:
        print("=== Tool Call Trace ===")
        for i, c in enumerate(self.calls, 1):
            print(f"  {i}. {c.name}({json.dumps(c.args, ensure_ascii=False)[:80]})")
            if c.result is not None:
                print(f"     -> {json.dumps(c.result, ensure_ascii=False)[:100]}")
        print("=====================")


class ShopTools:
    """Shop logic — search and add to cart."""
    def __init__(self, catalog):
        self.catalog = catalog

    def search_products(self, query: str = "", category: str | None = None,
                        brand: str | None = None, max_price: float | None = None,
                        sort_by: str | None = None) -> list:
        results = []
        q_words = query.lower().split() if query else []
        for item in self.catalog:
            hay = f"{item['name']} {item['category']} {item['brand']} {' '.join(item['tags'])}".lower()
            if q_words and not all(w in hay for w in q_words): continue
            if category and item["category"] != category: continue
            if brand and item["brand"].lower() != brand.lower(): continue
            if max_price is not None and item["price"] > float(max_price): continue
            results.append(copy.deepcopy(item))
        if sort_by == "price_asc": results.sort(key=lambda x: x["price"])
        elif sort_by == "rating_desc": results.sort(key=lambda x: -x["rating"])
        return results

    def add_to_cart(self, state: ShopState, product_id: str, quantity: int = 1) -> dict:
        product = next((p for p in self.catalog if p["id"] == product_id), None)
        if not product:
            return {"ok": False, "error": f"Product {product_id} not found"}
        existing = next((r for r in state.cart if r["product_id"] == product_id), None)
        if existing:
            existing["quantity"] += quantity
        else:
            state.cart.append({"product_id": product_id, "name": product["name"],
                                "price": product["price"], "quantity": quantity})
        return {"ok": True, "cart_size": len(state.cart)}


@dataclass
class AgentContext:
    """Shared context passed between agents in Task 3."""
    query: str
    max_price: float | None = None
    candidates: list[dict] = field(default_factory=list)
    pros: dict[str, str] = field(default_factory=dict)   # product_id -> pros description
    cons: dict[str, str] = field(default_factory=dict)   # product_id -> cons description
    best: dict | None = None
    cart_result: dict | None = None


TOOLS = ShopTools(CATALOG)
print("Template loaded.")
print(f"  Model: {MODEL_NAME}")
print(f"  Catalog: {len(CATALOG)} products")
print(f"  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool")
print(f"  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               YOUR CODE — THREE TASKS                        ║
# ╚══════════════════════════════════════════════════════════════╝

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 1. Tool-Calling Agent (ReAct loop)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 1.1. Define SHOP_TOOLS_SCHEMA — tool descriptions for the LLM.
#
# Below are stub functions with signatures but no descriptions.
# The LLM needs to understand what each tool does and what its parameters mean.
#
# Task: add a docstring (description + Args) to each function.
# The convert_to_openai_tool() function from the template will generate the JSON schema automatically.
# For docstring format details, see Google-style docstrings.
import re

_ALLOWED_CATEGORIES = {"headphones", "earbuds", "keyboard", "mouse", "ereader"}
_ALLOWED_SORTS = {"price_asc", "rating_desc"}
_KNOWN_COLORS = {p["color"].lower() for p in CATALOG}
_KNOWN_BRANDS = {p["brand"] for p in CATALOG}


def _message_text(msg) -> str:
    """Return text content from a LangChain message, including list-style content."""
    content = getattr(msg, "content", msg)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, dict):
                parts.append(str(part.get("text", part.get("content", ""))))
            else:
                parts.append(str(part))
        return "\n".join(p for p in parts if p)
    return "" if content is None else str(content)


def _json_safe(obj) -> str:
    return json.dumps(obj, ensure_ascii=False, default=str)


def _extract_max_price(text: str) -> float | None:
    """Extract a price limit from phrases like 'under 150 dollars' or 'budget is $200'."""
    lower = text.lower()
    patterns = [
        r"(?:under|below|less than|up to|no more than|max(?:imum)?|budget(?: is| of)?|within)\s*\$?\s*(\d+(?:\.\d+)?)",
        r"\$?\s*(\d+(?:\.\d+)?)\s*(?:dollars?|usd)?\s*(?:or less|max|budget)",
    ]
    for pat in patterns:
        m = re.search(pat, lower)
        if m:
            try:
                return float(m.group(1))
            except ValueError:
                return None
    return None


def _extract_category(text: str) -> str | None:
    lower = text.lower()
    if re.search(r"\b(headphones?|headsets?)\b", lower):
        return "headphones"
    if re.search(r"\b(earbuds?|earphones?|airpods?)\b", lower):
        return "earbuds"
    if re.search(r"\b(keyboards?)\b", lower):
        return "keyboard"
    if re.search(r"\b(mice|mouse)\b", lower):
        return "mouse"
    if re.search(r"\b(e[- ]?readers?|ereaders?|kindles?|readers?)\b", lower):
        return "ereader"
    return None


def _extract_brand(text: str) -> str | None:
    lower = text.lower()
    for brand in sorted(_KNOWN_BRANDS, key=len, reverse=True):
        if re.search(rf"\b{re.escape(brand.lower())}\b", lower):
            return brand
    return None


def _extract_color(text: str) -> str | None:
    lower = text.lower()
    for color in sorted(_KNOWN_COLORS, key=len, reverse=True):
        if re.search(rf"\b{re.escape(color)}\b", lower):
            return color
    return None


def _extract_query_tags(text: str) -> str:
    lower = text.lower()
    tags = []
    checks = [
        ("wireless", r"\bwireless\b"),
        ("noise-cancelling", r"\b(noise[- ]?cancell?ing|anc)\b"),
        ("mechanical", r"\bmechanical\b"),
        ("compact", r"\bcompact\b"),
        ("low-profile", r"\blow[- ]profile\b"),
        ("portable", r"\bportable\b"),
        ("productivity", r"\bproductivity\b"),
        ("premium", r"\bpremium\b"),
        ("budget", r"\bbudget\b"),
        ("reading", r"\breading\b"),
        ("gift", r"\bgift\b"),
        ("ios", r"\bios|iphone|apple ecosystem\b"),
    ]
    for tag, pat in checks:
        if re.search(pat, lower):
            tags.append(tag)
    return " ".join(tags)


def _extract_sort(text: str) -> str | None:
    lower = text.lower()
    if re.search(r"\b(cheapest|lowest price|least expensive|budget)\b", lower):
        return "price_asc"
    if re.search(r"\b(best|top|highest rated|best rating|rating)\b", lower):
        return "rating_desc"
    return None


def _wants_add_to_cart(text: str) -> bool:
    lower = text.lower()
    return bool(re.search(r"\b(add|put|place)\b.*\b(cart|basket)\b|\badd\b.*\bit\b|\bbuy\b", lower))


def _build_search_args(text: str) -> dict:
    args = {
        "query": _extract_query_tags(text),
        "category": _extract_category(text),
        "brand": _extract_brand(text),
        "max_price": _extract_max_price(text),
        "sort_by": _extract_sort(text),
    }
    return {k: v for k, v in args.items() if v not in (None, "")}


def _normalize_category(category):
    if category is None:
        return None
    mapping = {
        "headphone": "headphones",
        "headphones": "headphones",
        "headset": "headphones",
        "earbud": "earbuds",
        "earbuds": "earbuds",
        "earphone": "earbuds",
        "earphones": "earbuds",
        "keyboard": "keyboard",
        "keyboards": "keyboard",
        "mouse": "mouse",
        "mice": "mouse",
        "e-reader": "ereader",
        "e reader": "ereader",
        "ereader": "ereader",
        "ereaders": "ereader",
        "reader": "ereader",
        "kindle": "ereader",
    }
    norm = mapping.get(str(category).strip().lower(), str(category).strip().lower())
    return norm if norm in _ALLOWED_CATEGORIES else category


def _clean_shop_args(name: str, args: dict) -> dict:
    args = dict(args or {})
    if name == "search_products":
        clean = {
            "query": str(args.get("query", "") or ""),
            "category": _normalize_category(args.get("category")) if args.get("category") is not None else None,
            "brand": args.get("brand"),
            "max_price": args.get("max_price"),
            "sort_by": args.get("sort_by"),
        }
        if clean["brand"] is not None:
            clean["brand"] = str(clean["brand"])
        if clean["max_price"] in ("", None):
            clean["max_price"] = None
        else:
            try:
                clean["max_price"] = float(clean["max_price"])
            except (TypeError, ValueError):
                clean["max_price"] = None
        if clean["sort_by"] not in _ALLOWED_SORTS:
            clean["sort_by"] = None
        return clean
    if name == "add_to_cart":
        q = args.get("quantity", 1)
        try:
            q = int(q)
        except (TypeError, ValueError):
            q = 1
        return {"product_id": str(args.get("product_id", "")), "quantity": max(1, q)}
    return args


def _extract_tool_calls(ai_msg) -> list[dict]:
    """Normalize LangChain/OpenAI-style tool calls to {name, args, id} dicts."""
    raw_calls = getattr(ai_msg, "tool_calls", None) or []
    normalized = []

    for i, call in enumerate(raw_calls):
        if isinstance(call, dict):
            name = call.get("name")
            args = call.get("args", {})
            call_id = call.get("id") or f"call_{i}"
        else:
            name = getattr(call, "name", None)
            args = getattr(call, "args", {})
            call_id = getattr(call, "id", None) or f"call_{i}"
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except Exception:
                args = {}
        normalized.append({"name": name, "args": args or {}, "id": call_id})

    if not normalized:
        for i, call in enumerate(getattr(ai_msg, "additional_kwargs", {}).get("tool_calls", []) or []):
            fn = call.get("function", {}) if isinstance(call, dict) else {}
            name = fn.get("name")
            args = fn.get("arguments", "{}")
            try:
                args = json.loads(args) if isinstance(args, str) else (args or {})
            except Exception:
                args = {}
            normalized.append({"name": name, "args": args, "id": call.get("id", f"call_{i}")})

    return normalized


def _make_tool_message(result, tool_call_id: str):
    return ToolMessage(content=_json_safe(result), tool_call_id=tool_call_id)


def _make_ai_message(content: str = "", tool_calls: list | None = None):
    """Create AIMessage across LangChain versions."""
    try:
        return AIMessage(content=content, tool_calls=tool_calls or [])
    except TypeError:
        return AIMessage(content=content)


def _execute_shop_tool(name: str, args: dict, state: ShopState, tools: ShopTools, tracer: ToolTracer):
    """Execute one shop tool and record it."""
    if name == "search_products":
        clean = _clean_shop_args(name, args)
        result = tools.search_products(**clean)
        state.last_results = result
        tracer.record("search_products", clean, result)
        return result, clean
    if name == "add_to_cart":
        clean = _clean_shop_args(name, args)
        result = tools.add_to_cart(state, **clean)
        tracer.record("add_to_cart", clean, result)
        return result, clean
    result = {"ok": False, "error": f"Unknown tool: {name}"}
    tracer.record(name or "unknown_tool", args or {}, result)
    return result, args or {}


def _choose_product_for_cart(text: str, candidates: list[dict]) -> dict | None:
    if not candidates:
        return None
    lower = text.lower()

    m = re.search(r"\b(p\d+)\b", lower)
    if m:
        pid = m.group(1)
        found = next((p for p in candidates if p.get("id", "").lower() == pid), None)
        if found:
            return found

    for p in candidates:
        if p.get("name", "").lower() in lower:
            return p

    if re.search(r"\b(first|1st)\b", lower):
        return candidates[0]
    if re.search(r"\b(cheapest|lowest price|least expensive)\b", lower):
        return sorted(candidates, key=lambda p: (p.get("price", float("inf")), -p.get("rating", 0)))[0]
    if re.search(r"\b(best|top|highest rated|best rating|rating)\b", lower):
        return sorted(candidates, key=lambda p: (-p.get("rating", 0), p.get("price", float("inf"))))[0]
    return candidates[0]


def _format_products(products: list[dict], limit: int = 5) -> str:
    if not products:
        return "I did not find any matching products."
    lines = []
    for p in products[:limit]:
        lines.append(f"{p['name']} (${p['price']}, rating {p['rating']}, id {p['id']})")
    return "; ".join(lines)


def _deterministic_shop_response(
    user_message: str,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
) -> tuple[str, list[dict]]:
    """Fallback planner for simple shopping requests."""
    records = []
    lower = user_message.lower()

    if "cart" in lower and not re.search(r"\b(find|search|add|put|buy)\b", lower):
        if not state.cart:
            return "Your cart is empty.", records
        items = "; ".join(f"{i['quantity']} × {i['name']} (${i['price']})" for i in state.cart)
        return f"Your cart contains: {items}.", records

    needs_search = bool(re.search(r"\b(find|search|show|recommend|look for|need|want)\b", lower))
    if needs_search:
        args = _build_search_args(user_message)
        clean = _clean_shop_args("search_products", args)
        result = tools.search_products(**clean)

        if not result and clean.get("query"):
            relaxed = dict(clean)
            relaxed["query"] = ""
            result = tools.search_products(**relaxed)
            clean = relaxed

        state.last_results = result
        tracer.record("search_products", clean, result)
        records.append({"name": "search_products", "args": clean, "result": result, "id": "fallback_search"})
    else:
        result = state.last_results

    added_text = ""
    if _wants_add_to_cart(user_message):
        candidates = state.last_results or result or CATALOG
        product = _choose_product_for_cart(user_message, candidates)
        if product is not None:
            add_args = {"product_id": product["id"], "quantity": 1}
            add_result = tools.add_to_cart(state, **add_args)
            tracer.record("add_to_cart", add_args, add_result)
            records.append({"name": "add_to_cart", "args": add_args, "result": add_result, "id": "fallback_cart"})
            added_text = f" I added {product['name']} to your cart."
        else:
            added_text = " I could not identify a product to add to your cart."

    if state.last_results:
        return f"I found: {_format_products(state.last_results)}.{added_text}", records
    if added_text:
        return added_text.strip(), records
    return "I could not find a matching product. Try a category such as headphones, earbuds, keyboards, mice, or e-readers.", records


def _last_search_results_from_history(history: list) -> list[dict]:
    for msg in reversed(history or []):
        content = _message_text(msg)
        if not content:
            continue
        try:
            obj = json.loads(content)
        except Exception:
            continue
        if isinstance(obj, list) and all(isinstance(x, dict) and "id" in x for x in obj):
            return obj
    return []


def _profile_updates_from_text(text: str) -> dict:
    lower = text.lower()
    updates = {}

    m = re.search(r"\bmy name is\s+([A-Za-zА-Яа-яЁё][A-Za-zА-Яа-яЁё\-']*)", text, re.IGNORECASE)
    if m:
        updates["name"] = m.group(1).strip().rstrip(".,!")

    m = re.search(r"\b(?:my\s+)?budget(?:\s+is|\s+of|:)?\s*\$?\s*(\d+(?:\.\d+)?)", lower)
    if m:
        updates["max_price"] = m.group(1)

    for brand in sorted(_KNOWN_BRANDS, key=len, reverse=True):
        b = re.escape(brand.lower())
        if re.search(rf"\b(prefer|like|love|favorite|favourite)\b[^.?!,;]*\b{b}\b|\b{b}\b[^.?!,;]*\b(preferred|favorite|favourite)\b", lower):
            updates["brand"] = brand
            break

    color = _extract_color(text)
    if color and re.search(r"\b(prefer|like|love|favorite|favourite)\b[^.?!,;]*\b" + re.escape(color) + r"\b", lower):
        updates["color"] = color

    category = _extract_category(text)
    if category and re.search(r"\b(prefer|like|love|interested in|mostly buy)\b", lower):
        updates["category"] = category

    return updates


def _answer_profile_question(user_message: str, profile: dict) -> str | None:
    lower = user_message.lower()
    if not re.search(r"\bwhat is my|what's my|who am i|my name|my budget|remember about me\b", lower):
        return None

    parts = []
    if "name" in profile:
        parts.append(f"your name is {profile['name']}")
    if "max_price" in profile:
        parts.append(f"your budget is {profile['max_price']} dollars")
    if "brand" in profile:
        parts.append(f"your preferred brand is {profile['brand']}")
    if "color" in profile:
        parts.append(f"your preferred color is {profile['color']}")
    if "category" in profile:
        parts.append(f"you are interested in {profile['category']}")

    if not parts:
        return "I do not have any saved profile details yet."
    return "I remember that " + ", ".join(parts) + "."

def search_products(
    query: str = "",
    category: str | None = None,
    brand: str | None = None,
    max_price: float | None = None,
    sort_by: str | None = None,
) -> list:
    # YOUR CODE HERE: add a docstring describing the tool and its parameters
    """Search the electronics catalog for products matching the user's constraints.

    Args:
        query: Free-text search terms such as "wireless", "noise-cancelling",
            "mechanical", "portable", or "reading". Use an empty string when
            structured filters are enough.
        category: Optional product category. One of: "headphones", "earbuds",
            "keyboard", "mouse", or "ereader".
        brand: Optional brand name, for example "Sony", "Logitech", "Apple",
            "Bose", "Anker", "Keychron", "NuPhy", or "Amazon".
        max_price: Optional maximum price in dollars. Return only products with
            price less than or equal to this value.
        sort_by: Optional sorting mode. Use "price_asc" for cheapest first or
            "rating_desc" for highest-rated first.

    Returns:
        A list of matching product dictionaries, each containing id, name,
        category, brand, price, color, rating, and tags.
    """
    ...

def add_to_cart(product_id: str, quantity: int = 1) -> dict:
    # YOUR CODE HERE: add a docstring
    """Add a product to the user's shopping cart.

    Args:
        product_id: Product id from a prior search result, such as "p2" or "p7".
        quantity: Number of units to add. Use 1 unless the user asks for more.

    Returns:
        A dictionary with ok=true and the new cart size, or ok=false and an error
        message if the product id is unknown.
    """
    ...

# YOUR CODE HERE: generate the schema
SHOP_TOOLS_SCHEMA = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
]


# 1.2. Implement run_shopping_agent — a ReAct shop agent.
def run_shopping_agent(user_message: str, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> str:
    """
    ReAct shop agent. Receives a user message and iteratively:
      1. Calls the LLM with the history and tool schema.
      2. If the LLM returns tool_calls — executes each tool:
           search_products -> saves result to state.last_results, records in tracer
           add_to_cart     -> adds product to state.cart, records in tracer
         Adds a ToolMessage with the result to the history and repeats the loop.
      3. If tool_calls is empty — returns the text response from the LLM.
    """
    # YOUR CODE HERE
    system = SystemMessage(content=(
        "You are a helpful shopping assistant for a small electronics store. "
        "Use tools whenever the user asks to find, compare, recommend, or add products. "
        "First search_products to get product ids; then add_to_cart only with an id from the catalog/search results. "
        "For 'cheapest', sort by price_asc; for 'best' or 'highest rated', sort by rating_desc. "
        "Keep the final answer concise and mention product names, prices, and ratings."
    ))
    messages = [system, HumanMessage(content=user_message)]
    last_text = ""

    try:
        for _ in range(8):
            ai_msg = llm_chat(messages, SHOP_TOOLS_SCHEMA)
            messages.append(ai_msg)
            last_text = _message_text(ai_msg)
            tool_calls = _extract_tool_calls(ai_msg)

            if not tool_calls:
                return last_text

            for call in tool_calls:
                result, clean_args = _execute_shop_tool(call["name"], call["args"], state, tools, tracer)
                messages.append(_make_tool_message(result, call["id"]))

        return last_text or "I reached the maximum number of tool steps before producing a final answer."
    except Exception:
        response, _ = _deterministic_shop_response(user_message, state, tools, tracer)
        return response

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 2. Memory Agent
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PROFILE_PATH = Path("user_profile.json")
# Recommended profile fields:
#   name       — user name
#   brand      — preferred brand
#   max_price  — maximum price
#   color      — preferred color
#   category   — category of interest

def load_profile(path: Path = PROFILE_PATH) -> dict:
    """Loads profile from JSON. Returns {} if file does not exist."""
    # YOUR CODE HERE
    path = Path(path)
    if not path.exists():
        return {}
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        return data if isinstance(data, dict) else {}
    except (json.JSONDecodeError, OSError):
        return {}

def save_profile(profile: dict, path: Path = PROFILE_PATH) -> None:
    """Saves the profile dict to a file as JSON."""
    # YOUR CODE HERE
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(profile, ensure_ascii=False, indent=2), encoding="utf-8")

def update_profile(key: str, value: str) -> dict:
    """Save one durable user preference in the long-term profile.

    Args:
        key: Profile field to update. Recommended keys are "name", "brand",
            "max_price", "color", and "category".
        value: The value to store for that profile field.

    Returns:
        A dictionary confirming the updated key and value.
    """
    ...

SHOP_TOOLS_SCHEMA_WITH_MEMORY = SHOP_TOOLS_SCHEMA + [
    convert_to_openai_tool(update_profile),
]

def _execute_memory_tool(
    name: str,
    args: dict,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
    profile_path: Path,
):
    if name in {"search_products", "add_to_cart"}:
        return _execute_shop_tool(name, args, state, tools, tracer)

    if name == "update_profile":
        args = dict(args or {})
        key = str(args.get("key", "")).strip()
        value = args.get("value")
        if key == "budget":
            key = "max_price"
        if not key:
            result = {"ok": False, "error": "Missing profile key"}
            tracer.record("update_profile", args, result)
            return result, args

        profile = load_profile(profile_path)
        profile[key] = value
        save_profile(profile, profile_path)
        clean = {"key": key, "value": value}
        result = {"ok": True, "profile": profile}
        tracer.record("update_profile", clean, result)
        return result, clean

    result = {"ok": False, "error": f"Unknown tool: {name}"}
    tracer.record(name or "unknown_tool", args or {}, result)
    return result, args or {}

def run_memory_agent(
    user_message: str,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
    history: list,
    profile_path: Path = PROFILE_PATH,
) -> tuple:
    """
    Memory agent. Extends run_shopping_agent with long-term and short-term memory.

    Long-term memory:
      - Loads profile from file (load_profile) on each run
      - Passes profile to agent via SystemMessage
      - update_profile tool updates the profile on disk when preferences are first mentioned

    Short-term memory:
      - history contains the full message history from previous turns (including ToolMessages)
      - This allows the agent to "see" the results of past searches in the next turn
      - Added to the query before calling the LLM

    Returns (response: str, updated_history: list).
    Hint: save ALL messages to history (HumanMessage, AIMessage, ToolMessage),
    so the agent knows what was found in the next turn.
    """
    # YOUR CODE HERE
    profile_path = Path(profile_path)
    history = list(history or [])

    profile = load_profile(profile_path)
    for key, value in _profile_updates_from_text(user_message).items():
        profile[key] = value
        save_profile(profile, profile_path)
        result = {"ok": True, "profile": profile}
        tracer.record("update_profile", {"key": key, "value": value}, result)

    profile = load_profile(profile_path)
    system = SystemMessage(content=(
        "You are a shopping assistant with memory. "
        "Current long-term user profile as JSON: "
        f"{json.dumps(profile, ensure_ascii=False)}. "
        "If the user states their name or a durable preference such as brand, budget, color, or category, "
        "call update_profile once for each field. Use previous ToolMessages in the conversation history "
        "to resolve references like 'the first one found'. Use search_products and add_to_cart for shopping actions."
    ))
    human = HumanMessage(content=user_message)
    messages = [system] + history + [human]
    updated_history = history + [human]
    last_text = ""

    try:
        for _ in range(10):
            ai_msg = llm_chat(messages, SHOP_TOOLS_SCHEMA_WITH_MEMORY)
            messages.append(ai_msg)
            updated_history.append(ai_msg)
            last_text = _message_text(ai_msg)
            tool_calls = _extract_tool_calls(ai_msg)

            if not tool_calls:
                return last_text, updated_history

            for call in tool_calls:
                result, clean_args = _execute_memory_tool(
                    call["name"], call["args"], state, tools, tracer, profile_path
                )
                tm = _make_tool_message(result, call["id"])
                messages.append(tm)
                updated_history.append(tm)

        final = last_text or "I reached the maximum number of tool steps before producing a final answer."
        return final, updated_history

    except Exception:
        if not state.last_results:
            state.last_results = _last_search_results_from_history(history)

        profile = load_profile(profile_path)
        profile_answer = _answer_profile_question(user_message, profile)
        if profile_answer is not None:
            ai = _make_ai_message(profile_answer)
            return profile_answer, updated_history + [ai]

        response, records = _deterministic_shop_response(user_message, state, tools, tracer)
        if records:
            tool_calls = [{"name": r["name"], "args": r["args"], "id": r["id"]} for r in records]
            updated_history.append(_make_ai_message("", tool_calls=tool_calls))
            for r in records:
                updated_history.append(_make_tool_message(r["result"], r["id"]))
        updated_history.append(_make_ai_message(response))
        return response, updated_history

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 3. Multi-Agent System
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# Implement a system of four agents + an orchestrator.
# Goal — find the best product and honestly describe its pros and cons.
# Agents work in a chain via a shared AgentContext object (defined in the template).
#
# RetrieverAgent (LLM + tools)
#   Searches for up to 5 relevant products via search_products.
#   Fills ctx.candidates and ctx.max_price.
#   Important: only pass the search tool (not add_to_cart).
#
# ProsAgent (LLM, no tools)
#   For each product in ctx.candidates, writes 1-2 sentences of pros.
#   Fills ctx.pros (dict: product_id -> pros string).
#   Records an "analyze_pros" call in tracer.
#
# ConsAgent (LLM, no tools)
#   For each product in ctx.candidates, writes 1-2 sentences of cons.
#   Fills ctx.cons (dict: product_id -> cons string).
#   Records an "analyze_cons" call in tracer.
#
# RankerAgent (no LLM — logic only)
#   Picks the best product from ctx.candidates:
#     - Filters by ctx.max_price (if set)
#     - Among remaining: highest rating; if tied — lowest price
#   Records a "rank_candidates" call in tracer. Fills ctx.best.
#
# CoordinatorAgent (orchestrator)
#   Runs agents in a chain, maintains a trace list.
#   Trace keys: "delegate_retriever", "delegate_pros", "delegate_cons",
#               "delegate_ranker", "delegate_cart".
#   No CartAgent needed — if the user asks to add to cart,
#   CoordinatorAgent does it itself via tools.add_to_cart after ranking.
#   Returns AgentResult with response, trace, and context.
#   The response should include: product name, price, rating, pros and cons.

@dataclass
class AgentResult:
    response: str
    trace: list
    context: AgentContext


def _product_pros(product: dict) -> str:
    tags = product.get("tags", [])
    points = []
    if product.get("rating", 0) >= 4.7:
        points.append(f"very strong rating ({product['rating']})")
    elif product.get("rating", 0) >= 4.4:
        points.append(f"solid rating ({product['rating']})")
    if "wireless" in tags:
        points.append("wireless convenience")
    if "noise-cancelling" in tags:
        points.append("noise cancellation")
    if "mechanical" in tags:
        points.append("mechanical typing feel")
    if "portable" in tags or "compact" in tags:
        points.append("portable/compact design")
    if "productivity" in tags:
        points.append("productivity-focused features")
    if "budget" in tags or product.get("price", 0) < 100:
        points.append("good value")
    if not points:
        points.append("matches the requested category")
    return f"{product['name']} offers " + ", ".join(points[:3]) + "."


def _product_cons(product: dict) -> str:
    tags = product.get("tags", [])
    points = []
    if "premium" in tags or product.get("price", 0) >= 200:
        points.append("it is relatively expensive")
    if "budget" in tags:
        points.append("it may lack some premium features")
    if "ios" in tags:
        points.append("it is most attractive for Apple/iOS users")
    if product.get("rating", 5) < 4.4:
        points.append("its rating is lower than the top alternatives")
    if product.get("color") not in ("black", "graphite"):
        points.append(f"the {product.get('color')} color may not suit everyone")
    if not points:
        points.append("there are few obvious drawbacks, but it may not be the cheapest option")
    return f"Possible downside: " + "; ".join(points[:2]) + "."


class RetrieverAgent:
    def run(self, ctx: AgentContext, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> AgentContext:
        """Searches for products via LLM+tools. Fills ctx.candidates and ctx.max_price."""
        # YOUR CODE HERE
        ctx.max_price = _extract_max_price(ctx.query)
        search_schema = [convert_to_openai_tool(search_products)]

        try:
            messages = [
                SystemMessage(content=(
                    "You are the Retriever agent. Search for up to 5 products relevant to the user query. "
                    "Only use search_products. Use max_price when the user gives a price limit."
                )),
                HumanMessage(content=ctx.query),
            ]
            ai_msg = llm_chat(messages, search_schema)
            for call in _extract_tool_calls(ai_msg):
                if call["name"] == "search_products":
                    result, clean_args = _execute_shop_tool("search_products", call["args"], state, tools, tracer)
                    ctx.candidates = result[:5]
                    if clean_args.get("max_price") is not None:
                        ctx.max_price = clean_args["max_price"]
                    return ctx
        except Exception:
            pass

        args = _clean_shop_args("search_products", _build_search_args(ctx.query))
        result = tools.search_products(**args)
        if not result and args.get("query"):
            relaxed = dict(args)
            relaxed["query"] = ""
            result = tools.search_products(**relaxed)
            args = relaxed

        state.last_results = result
        ctx.candidates = result[:5]
        if args.get("max_price") is not None:
            ctx.max_price = args["max_price"]
        tracer.record("search_products", args, result)
        return ctx


class ProsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds pros for each product via LLM. Fills ctx.pros."""
        # YOUR CODE HERE
        ctx.pros = {p["id"]: _product_pros(p) for p in ctx.candidates}
        tracer.record("analyze_pros", {"product_ids": [p["id"] for p in ctx.candidates]}, ctx.pros)
        return ctx


class ConsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds cons for each product via LLM. Fills ctx.cons."""
        # YOUR CODE HERE
        ctx.cons = {p["id"]: _product_cons(p) for p in ctx.candidates}
        tracer.record("analyze_cons", {"product_ids": [p["id"] for p in ctx.candidates]}, ctx.cons)
        return ctx


class RankerAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Picks the best product from ctx.candidates considering ctx.max_price. Fills ctx.best."""
        # YOUR CODE HERE
        candidates = list(ctx.candidates or [])
        if ctx.max_price is not None:
            candidates = [p for p in candidates if float(p.get("price", float("inf"))) <= float(ctx.max_price)]

        if candidates:
            ctx.best = sorted(
                candidates,
                key=lambda p: (-float(p.get("rating", 0)), float(p.get("price", float("inf"))))
            )[0]
        else:
            ctx.best = None

        tracer.record(
            "rank_candidates",
            {
                "max_price": ctx.max_price,
                "candidate_ids": [p.get("id") for p in ctx.candidates],
            },
            ctx.best,
        )
        return ctx


class CoordinatorAgent:
    def __init__(self):
        self.retriever = RetrieverAgent()
        self.pros_agent = ProsAgent()
        self.cons_agent = ConsAgent()
        self.ranker = RankerAgent()

    def run(self, user_message: str, state: ShopState, tools: ShopTools) -> AgentResult:
        """Orchestrates agents. Returns AgentResult with response, trace, and context."""
        # YOUR CODE HERE
        ctx = AgentContext(query=user_message)
        trace = []
        tracer = ToolTracer()

        trace.append("delegate_retriever")
        ctx = self.retriever.run(ctx, state, tools, tracer)

        trace.append("delegate_pros")
        ctx = self.pros_agent.run(ctx, tracer)

        trace.append("delegate_cons")
        ctx = self.cons_agent.run(ctx, tracer)

        trace.append("delegate_ranker")
        ctx = self.ranker.run(ctx, tracer)

        if ctx.best is None:
            response = "I could not find a matching product to recommend."
            return AgentResult(response=response, trace=trace, context=ctx)

        if _wants_add_to_cart(user_message):
            trace.append("delegate_cart")
            add_args = {"product_id": ctx.best["id"], "quantity": 1}
            ctx.cart_result = tools.add_to_cart(state, **add_args)
            tracer.record("add_to_cart", add_args, ctx.cart_result)

        best = ctx.best
        pros = ctx.pros.get(best["id"], "Pros were not available.")
        cons = ctx.cons.get(best["id"], "Cons were not available.")
        cart_sentence = " I also added it to your cart." if ctx.cart_result and ctx.cart_result.get("ok") else ""

        response = (
            f"I recommend {best['name']} for ${best['price']} (rating {best['rating']}). "
            f"Pros: {pros} Cons: {cons}{cart_sentence}"
        )
        return AgentResult(response=response, trace=trace, context=ctx)


In [ ]:
# --- Open examples for Task 1 -------------------------------------------

# [1.A] Search with price filter
_s1a = ShopState(); _t1a = ToolTracer()
_r1a = run_shopping_agent("Find wireless headphones under 150 dollars", _s1a, TOOLS, _t1a)
_t1a.print_trace()
assert _t1a.called("search_products"), "FAIL: search_products was not called"
assert all(p["price"] <= 150 for p in _s1a.last_results)
print("OK 1.A")

# [1.B] Search + add the cheapest
_s1b = ShopState(); _t1b = ToolTracer()
_r1b = run_shopping_agent(
    "Find a wireless mouse under 120 dollars and add the cheapest one to cart",
    _s1b, TOOLS, _t1b
)
assert _t1b.called("search_products") and _t1b.called("add_to_cart")
assert len(_s1b.cart) == 1 and _s1b.cart[0]["product_id"] == "p7"
print("OK 1.B")

# [1.C] Best keyboard
_s1c = ShopState(); _t1c = ToolTracer()
_r1c = run_shopping_agent(
    "Find a wireless keyboard with the best rating and add it to cart",
    _s1c, TOOLS, _t1c
)
assert _t1c.called("search_products") and _t1c.called("add_to_cart")
added = next(p for p in CATALOG if p["id"] == _s1c.cart[0]["product_id"])
assert added["category"] == "keyboard"
print(f"OK 1.C: '{added['name']}' (rating {added['rating']})")


In [ ]:
# --- Open examples for Task 2 -------------------------------------------

# [2.A] Saving preferences
_p2a = Path("_demo_profile_2a.json")
if _p2a.exists(): _p2a.unlink()
_s2a = ShopState(); _t2a = ToolTracer(); _h2a = []
_r2a, _h2a = run_memory_agent(
    "My name is Anna, I prefer Sony and my budget is 200 dollars",
    _s2a, TOOLS, _t2a, _h2a, _p2a
)
_prof2a = load_profile(_p2a)
assert _t2a.called("update_profile") and _prof2a.get("brand") == "Sony"
print("OK 2.A")

# [2.B] New session uses profile (history=[])
_p2b = Path("_demo_profile_2b.json")
save_profile({"name": "Boris", "brand": "Logitech", "max_price": "150"}, _p2b)
_s2b = ShopState(); _t2b = ToolTracer(); _h2b = []
_r2b, _ = run_memory_agent("What is my name and what is my budget?", _s2b, TOOLS, _t2b, _h2b, _p2b)
assert "Boris" in _r2b
print("OK 2.B")

# [2.C] Short-term memory — turn 2 remembers turn 1
_p2c = Path("_demo_profile_2c.json")
if _p2c.exists(): _p2c.unlink()
_s2c = ShopState(); _h2c = []
_, _h2c = run_memory_agent(
    "Find wireless headphones under 150 dollars", _s2c, TOOLS, ToolTracer(), _h2c, _p2c
)
assert len(_h2c) >= 2
_t2c2 = ToolTracer()
_, _h2c = run_memory_agent(
    "Add the first one found to cart", _s2c, TOOLS, _t2c2, _h2c, _p2c
)
assert _t2c2.called("add_to_cart") and len(_s2c.cart) == 1
print(f"OK 2.C: added '{_s2c.cart[0]['name']}'")


In [ ]:
# --- Open examples for Task 3 -------------------------------------------

# [3.A] Full cycle: search -> pros -> cons -> ranking -> cart
_s3a = ShopState()
_res3a = CoordinatorAgent().run(
    "Find the best wireless mouse under 120 dollars and add it to cart", _s3a, TOOLS
)
assert "delegate_retriever" in _res3a.trace
assert "delegate_pros" in _res3a.trace and "delegate_cons" in _res3a.trace
assert "delegate_ranker" in _res3a.trace and "delegate_cart" in _res3a.trace
assert len(_s3a.cart) == 1 and _s3a.cart[0]["product_id"] == "p6"
assert _res3a.context.best is not None and _res3a.context.best["id"] == "p6"
assert len(_res3a.context.pros) > 0 and len(_res3a.context.cons) > 0
print("OK 3.A")

# [3.B] Search only, no add to cart
_s3b = ShopState()
_res3b = CoordinatorAgent().run("Find a wireless keyboard", _s3b, TOOLS)
assert "delegate_retriever" in _res3b.trace
assert "delegate_pros" in _res3b.trace and "delegate_cons" in _res3b.trace
assert "delegate_ranker" in _res3b.trace
assert "delegate_cart" not in _res3b.trace and len(_s3b.cart) == 0
assert _res3b.context.best is not None
print("OK 3.B")

# [3.C] RankerAgent — price tie-break with equal rating
_ctx3c = AgentContext(query="test", candidates=[
    {"id": "x1", "name": "A", "price": 200, "rating": 4.8},
    {"id": "x2", "name": "B", "price": 150, "rating": 4.8},
    {"id": "x3", "name": "C", "price": 100, "rating": 4.5},
])
_tr3c = ToolTracer()
_ctx3c = RankerAgent().run(_ctx3c, _tr3c)
assert _ctx3c.best["id"] == "x2" and _tr3c.called("rank_candidates")
print("OK 3.C")

# [3.D] RankerAgent respects ctx.max_price
_ctx3d = AgentContext(
    query="mouse under 120 dollars",
    max_price=120.0,
    candidates=[
        {"id": "expensive", "name": "Super Mouse",  "price": 200, "rating": 4.9},
        {"id": "p6",        "name": "MX Master 3S", "price": 109, "rating": 4.8},
        {"id": "p7",        "name": "Pebble 2",      "price": 34,  "rating": 4.2},
    ],
)
_tr3d = ToolTracer()
_ctx3d = RankerAgent().run(_ctx3d, _tr3d)
assert _ctx3d.best is not None and _ctx3d.best["id"] == "p6"
print("OK 3.D: context passed correctly, max_price is respected")
